# HK Estate: explore the growth study

Reproduces the key tables of `hkestate report` (data/reports/findings_<date>.md) from the database. Run `hkestate train` first so estimated pairs use the current model.

Reading the database takes about 40 seconds and needs it free for that time; set `FROM_SNAPSHOT = True` to reuse `data/model/snapshot` instead. Outputs are stripped on commit (nbstripout).

In [ ]:
FROM_SNAPSHOT = False  # True: reuse data/model/snapshot (no database read)

In [ ]:
import pandas as pd

from hkestate import features, model, study

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)

raw = features.load_snapshot() if FROM_SNAPSHOT else features.snapshot()
m = model.load()
tp = pd.read_parquet(model.model_dir() / 'test_predictions.parquet')
s = study.run(raw, m, tp)
raw['meta']

## Pairs by type

In [ ]:
pd.Series({k: len(v) for k, v in s.pairs.items()}, name='pairs')

## Verdicts: recorded pairs (real + Ricacorp)

`coef` is annual log growth (x100 = percentage points a year). Robust = p < 0.05 and the same sign in both halves of the time and district splits.

In [ ]:
cols = ['label', 'coef', 'se', 'p', 'n_with', 'time_a', 'time_b', 'district_a',
        'district_b', 'robust']
s.verdicts['recorded'][cols]

## Verdicts: estimated pairs (pre-2018 sale vs model value today)

In [ ]:
# needs 200+ estimated pairs
s.verdicts.get('estimated', pd.DataFrame()).reindex(columns=cols + ['model_tilt'])

## Growth, costs, rent and total return (summary by pair type)

In [ ]:
pd.concat(s.summaries, names=['pairs', 'measure'])

## First-hand vs second-hand

In [ ]:
s.first_hand

## Drawdown by group

In [ ]:
s.drawdowns

## Long-run context (RVD)

In [ ]:
s.rvd_long

## Checks: % change pairs and model error by attribute

In [ ]:
s.gain_check, s.gain_signs, s.model_bias

## Coverage

In [ ]:
s.coverage['by_source'], s.coverage['exclusions']